## 5. Односвязный список (Singly Linked List)

**Односвязный список** состоит из узлов. Каждый узел хранит значение и ссылку на следующий узел. Последний узел ссылается на `None`.

Достоинства:
- вставка/удаление в начале за **O(1)**;
- нет необходимости заранее знать размер;
- удобно строить очереди и стеки поверх узлов.

Недостатки:
- доступ по индексу за **O(n)**;
- нельзя быстро идти назад.

**Где применяется:** плейлист «дальше», цепочки задач, списки смежности в графах, реализации очередей.

In [13]:
class SNode:
    # __slots__ = ('value', 'next')

    def __init__(self, value, next=None):
        self._value = value
        self._next = next

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, value):
        self._value = value

    @property
    def next(self):
        return self._next

    @next.setter
    def next(self, next):
        self._next = next

    def __repr__(self):
        return f'SNode({self.value!r})'

class SinglyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def get_head(self):
        return self.head

    def get_tail(self):
        return self.tail

    def append(self, value):
        node = SNode(value)
        if self.head is None:
            self.head = node
            self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._size += 1

    def prepend(self, value):
        node = SNode(value, self.head)
        self.head = node
        if self.tail is None:
            self.tail = node
        self._size += 1

    def insert_at(self, index, value):
        if index < 0 or index > self._size:
            raise IndexError('index out of range')
        if index == 0:
            self.prepend(value)
            return
        if index == self._size:
            self.append(value)
            return
        current = self.head
        for _ in range(index - 1):
            current = current.next
        current.next = SNode(value, current.next)
        self._size += 1

    def delete_at(self, index):
        if index < 0 or index >= self._size:
            raise IndexError('index out of range')
        if index == 0:
            value = self.head.value
            self.head = self.head.next
            if self.head is None:
                self.tail = None
            self._size -= 1
            return value
        prev = self.head
        for _ in range(index - 1):
            prev = prev.next
        target = prev.next
        prev.next = target.next
        if target is self.tail:
            self.tail = prev
        self._size -= 1
        return target.value

    def remove(self, value):
        prev = None
        current = self.head
        while current is not None:
            if current.value == value:
                if prev is None:
                    self.head = current.next
                else:
                    prev.next = current.next
                if current is self.tail:
                    self.tail = prev
                self._size -= 1
                return True
            prev = current
            current = current.next
        return False

    def search(self, value):
        index = 0
        current = self.head
        while current is not None:
            if current.value == value:
                return index
            current = current.next
            index += 1
        return -1

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.value
            current = current.next

    def __len__(self):
        return self._size

    def __repr__(self):
        return 'SinglyLinkedList([' + ', '.join(repr(x) for x in self) + '])'

ll = SinglyLinkedList()
ll.append(1)
ll.append(2)
ll.prepend(0)
ll.insert_at(2, 99)
assert list(ll) == [0, 1, 99, 2]
assert ll.delete_at(2) == 99
assert ll.remove(1) is True
assert list(ll) == [0, 2]
assert ll.search(2) == 1
assert len(ll) == 2

### Практический пример: плейлист и разворот списка

Плейлист с кнопкой «Next» естественно моделируется односвязным списком: у нас есть только движение вперед.

Также полезно уметь разворачивать односвязный список за **O(n)** и **O(1)** дополнительной памяти.

In [ ]:
class Playlist:
    def __init__(self):
        self.tracks = SinglyLinkedList()
        self.current = None

    def add(self, title):
        self.tracks.append(title)

    def start(self):
        self.current = self.tracks.head

    def play_next(self):
        if self.current is None:
            return None
        title = self.current.value
        self.current = self.current.next
        return title

pl = Playlist()
pl.add('Song A')
pl.add('Song B')
pl.add('Song C')
pl.start()
assert pl.play_next() == 'Song A'
assert pl.play_next() == 'Song B'
assert pl.play_next() == 'Song C'
assert pl.play_next() is None

def reverse_singly_linked_list(ll):
    old_head = ll.head
    prev = None
    current = ll.head
    while current is not None:
        nxt = current.next
        current.next = prev
        prev = current
        current = nxt
    ll.head = prev
    ll.tail = old_head
    return ll

ll2 = SinglyLinkedList()
for x in [1, 2, 3, 4]:
    ll2.append(x)
reverse_singly_linked_list(ll2)
assert list(ll2) == [4, 3, 2, 1]
assert ll2.tail.value == 1

# Yield 

In [18]:
def f(start = 0, step=1):
    i = start
    while True:
        yield i
        i += step

In [19]:
g = f()

In [21]:
next(g)

0

In [22]:
next(g)

1

In [23]:
next(g)

2

## 6. Двусвязный список (Doubly Linked List)

**Двусвязный список** хранит в каждом узле две ссылки: `prev` и `next`. Это позволяет ходить в обе стороны и удалять известный узел за **O(1)**, если у нас есть ссылка на него.

Достоинства:
- обход вперед и назад;
- быстрое удаление узла при наличии ссылки;
- удобен для историй, плейлистов с Prev/Next, LRU-кэша.

Недостатки:
- больше памяти на каждую ссылку;
- больше случаев для ошибок при изменении указателей.

**Где применяется:** история браузера, навигация по документу, кэш с вытеснением давно используемых элементов.

In [24]:
class DNode:
    __slots__ = ('value', 'prev', 'next')

    def __init__(self, value, prev=None, next=None):
        self.value = value
        self.prev = prev
        self.next = next

    def __repr__(self):
        return f'DNode({self.value!r})'

class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def append(self, value):
        node = DNode(value, prev=self.tail)
        if self.head is None:
            self.head = node
        else:
            self.tail.next = node
        self.tail = node
        self._size += 1

    def prepend(self, value):
        node = DNode(value, next=self.head)
        if self.head is None:
            self.tail = node
        else:
            self.head.prev = node
        self.head = node
        self._size += 1

    def insert_at(self, index, value):
        if index < 0 or index > self._size:
            raise IndexError('index out of range')
        if index == 0:
            self.prepend(value)
            return
        if index == self._size:
            self.append(value)
            return
        current = self.head
        for _ in range(index):
            current = current.next
        node = DNode(value, prev=current.prev, next=current)
        current.prev.next = node
        current.prev = node
        self._size += 1

    def delete_at(self, index):
        if index < 0 or index >= self._size:
            raise IndexError('index out of range')
        current = self.head
        for _ in range(index):
            current = current.next
        value = current.value
        self._unlink(current)
        return value

    def remove(self, value):
        current = self.head
        while current is not None:
            if current.value == value:
                self._unlink(current)
                return True
            current = current.next
        return False

    def _unlink(self, node):
        if node.prev is None:
            self.head = node.next
        else:
            node.prev.next = node.next
        if node.next is None:
            self.tail = node.prev
        else:
            node.next.prev = node.prev
        node.prev = None
        node.next = None
        self._size -= 1

    def add_front_node(self, node):
        node.prev = None
        node.next = self.head
        if self.head is None:
            self.tail = node
        else:
            self.head.prev = node
        self.head = node
        self._size += 1

    def move_node_to_front(self, node):
        if node is self.head:
            return
        self._unlink(node)
        self.add_front_node(node)

    def remove_after(self, node):
        current = node.next
        while current is not None:
            nxt = current.next
            self._unlink(current)
            current = nxt

    def pop_back(self):
        if self.tail is None:
            raise IndexError('pop from empty list')
        value = self.tail.value
        self._unlink(self.tail)
        return value

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.value
            current = current.next

    def __reversed__(self):
        current = self.tail
        while current is not None:
            yield current.value
            current = current.prev

    def __len__(self):
        return self._size

    def __repr__(self):
        return 'DoublyLinkedList([' + ', '.join(repr(x) for x in self) + '])'

dl = DoublyLinkedList()
dl.append('A')
dl.append('B')
dl.prepend('C')
assert list(dl) == ['C', 'A', 'B']
assert list(reversed(dl)) == ['B', 'A', 'C']
dl.move_node_to_front(dl.tail)
assert list(dl) == ['B', 'C', 'A']
assert dl.pop_back() == 'A'
assert list(dl) == ['B', 'C']

### Практический пример: история браузера и LRU-кэш

История браузера — это двусвязный список плюс указатель `current`:
- `visit` удаляет всю «будущую» историю после текущей страницы и добавляет новую;
- `back` и `forward` двигают указатель по ссылкам `prev`/`next`.

LRU-кэш (Least Recently Used) сочетает словарь `key -> node` и двусвязный список порядка использования. Самый свежий ключ — в голове, самый старый — в хвосте.

In [25]:
class BrowserHistory:
    def __init__(self, homepage):
        self.history = DoublyLinkedList()
        self.history.append(homepage)
        self.current = self.history.head

    def visit(self, url):
        self.history.remove_after(self.current)
        self.history.append(url)
        self.current = self.history.tail

    def back(self, steps):
        for _ in range(steps):
            if self.current.prev is None:
                break
            self.current = self.current.prev
        return self.current.value

    def forward(self, steps):
        for _ in range(steps):
            if self.current.next is None:
                break
            self.current = self.current.next
        return self.current.value

bh = BrowserHistory('yandex.ru')
bh.visit('wikipedia.org')
bh.visit('python.org')
assert bh.back(1) == 'wikipedia.org'
bh.visit('github.com')
assert bh.forward(1) == 'github.com'
assert bh.back(2) == 'yandex.ru'

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.nodes = {}
        self.order = DoublyLinkedList()

    def get(self, key):
        if key not in self.nodes:
            return None
        node = self.nodes[key]
        self.order.move_node_to_front(node)
        return node.value[1]

    def put(self, key, value):
        if self.capacity <= 0:
            return
        if key in self.nodes:
            node = self.nodes[key]
            node.value = (key, value)
            self.order.move_node_to_front(node)
            return
        if len(self.nodes) == self.capacity:
            evicted = self.order.pop_back()
            del self.nodes[evicted[0]]
        node = DNode((key, value))
        self.order.add_front_node(node)
        self.nodes[key] = node

cache = LRUCache(2)
cache.put('a', 1)
cache.put('b', 2)
assert cache.get('a') == 1
cache.put('c', 3)
assert cache.get('b') is None
assert cache.get('a') == 1
assert cache.get('c') == 3

## 7. Сравнение структур данных

| Структура | Принцип | Добавление | Удаление | Произвольный доступ |
|---|---|---|---|---|
| Стек | LIFO | `push` O(1) | `pop` O(1) | только верхушка |
| Очередь | FIFO | `enqueue` O(1) | `dequeue` O(1) | только голова |
| Дек | оба конца | O(1) слева/справа | O(1) слева/справа | только концы |
| Односвязный список | `next` | O(1) в начало, O(1) в конец при `tail` | O(1) в начало, O(n) по значению | O(n) |
| Двусвязный список | `prev` и `next` | O(1) в оба конца | O(1) при ссылке на узел | O(n) |

**Важно:** асимптотика зависит от реализации. Например, `list.pop(0)` в Python — это **O(n)**, поэтому для очереди лучше `deque` или связный список.

## 8. Задачи для самостоятельного закрепления

Реализуйте классы/функции и добавьте тесты через `assert`.

1. **MinStack**: стек с методом `get_min()`, который возвращает минимальный элемент за O(1).
2. **QueueFromTwoStacks**: очередь, использующая только два стека.
3. **CircularQueue**: кольцевая очередь фиксированного размера на списке/массиве.
4. **PalindromeDeque**: проверить палиндром, игнорируя пробелы, пунктуацию и регистр, используя дек.
5. **MergeSortedLists**: объединить два отсортированных односвязных списка в один отсортированный за O(n).
6. **HasCycle**: обнаружить цикл в односвязном списке алгоритмом Флойда (черепаха и заяц).
7. **DequeFromDoublyLinkedList**: реализовать дек поверх собственного двусвязного списка без `collections.deque`.
8. **PlaylistPrevNext**: плейлист с методами `next`, `prev`, `jump_to(index)`, используя двусвязный список.
9. **LRUWithTimestamp**: расширить LRU-кэш так, чтобы `get` и `put` обновляли время обращения, а метод `dump()` возвращал ключи от самых свежих к самым старым.
10. **UndoRedoBrowser**: объединить редактор с undo/redo и историю браузера в один класс `Workspace`.

In [27]:
import time
time.time()

1791356547.9614894

In [28]:
time.time()

1791356620.2394886

In [ ]:
class MinStack(Stack):
    def __init__(self):
        super().__init__()
        self._mins = []

    def push(self, item):
        # TODO: добавьте элемент и обновите вспомогательный стек минимумов
        raise NotImplementedError

    def pop(self):
        # TODO: удалите элемент и, если он был минимумом, обновите минимум
        raise NotImplementedError

    def get_min(self):
        # TODO: вернуть текущий минимум за O(1)
        raise NotImplementedError

# После реализации должны пройти проверки:
# m = MinStack()
# m.push(3)
# m.push(5)
# m.push(2)
# assert m.get_min() == 2
# m.pop()
# assert m.get_min() == 3

class QueueFromTwoStacks:
    def __init__(self):
        self.in_stack = Stack()
        self.out_stack = Stack()

    def enqueue(self, item):
        # TODO: просто кладите элемент в in_stack
        raise NotImplementedError

    def dequeue(self):
        # TODO: если out_stack пуст, переложите все элементы из in_stack в out_stack
        raise NotImplementedError

# TODO: реализуйте CircularQueue по аналогии с Queue, но с полями head, tail, size, data

def merge_sorted_lists(l1, l2):
    # TODO: объедините два отсортированных SinglyLinkedList в новый список
    raise NotImplementedError

def has_cycle(head):
    # TODO: алгоритм Флойда: медленный и быстрый указатели
    raise NotImplementedError

class DequeFromDLL:
    def __init__(self):
        self.data = DoublyLinkedList()

    def add_left(self, item):
        self.data.prepend(item)

    def add_right(self, item):
        self.data.append(item)

    def pop_left(self):
        if self.data.head is None:
            raise IndexError('pop from empty deque')
        return self.data.delete_at(0)

    def pop_right(self):
        if self.data.tail is None:
            raise IndexError('pop from empty deque')
        return self.data.pop_back()

# Этот шаблон уже должен работать поверх DoublyLinkedList:
dd = DequeFromDLL()
dd.add_right(2)
dd.add_left(1)
dd.add_right(3)
assert dd.pop_left() == 1
assert dd.pop_right() == 3

## 9. Итоги и что читать дальше

Вы реализовали ключевые линейные структуры данных через ООП. Главное правило выбора:
- нужен только последний элемент — **стек**;
- нужен первый вошедший — **очередь**;
- нужны оба конца — **дек**;
- важна вставка/удаление в середине при наличии ссылки — **двусвязный список**;
- достаточно движения вперед и экономия памяти — **односвязный список**.

**Дальше:** изучите `collections.deque`, `heapq`, словари и множества, деревья и графы. Попробуйте переписать очередь и дек на собственных связных списках и сравнить скорость с встроенными средствами.